# Twist Unified Intake Converter · 2.01
Click ▶ once, then drop one or multiple **XLSX/CSV files** into the upload area or choose files. Conversion starts automatically.
The v3.1 output uses the supplied template, preserving its layout and filling only occupied source Order Details fields. Review warnings before submitting an order.


In [ ]:
# @title ▶ Start unified converter { display-mode: "form" }
import subprocess, sys, tempfile
from pathlib import Path
from IPython.display import clear_output
print('Preparing converter…')
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'openpyxl==3.1.5', 'lxml>=5,<7'], check=True, capture_output=True, text=True)
except subprocess.CalledProcessError as exc:
    raise RuntimeError('Setup failed. Reconnect the runtime and try again. ' + exc.stderr[-1500:]) from exc
RELEASE_SOURCES = {'assets': "import base64,zlib\nEC_TEMPLATES = zlib.decompress(base64.b64decode(\n    '...'\n))\nV42_TEMPLATE = zlib.decompress(base64.b64decode(\n    '...'\n))\n", 'ec_engine': 'import io,re,zipfile,os,collections,copy,json,csv\nimport openpyxl\nfrom openpyxl.utils import get_column_letter\n\ndef norm(s): return re.sub(r\'[^a-z0-9]+\',\'\',str(s or \'\').lower())\ndef populated(v): return v is not None and str(v).strip() and not str(v).lstrip().startswith(\'=\') and \'PLEASE SELECT OPTION\' not in str(v).upper()\ndef seq(v): return str(v).strip() if populated(v) else None\n\ndef find_input(wb):\n    # Contents take precedence over worksheet names (v4.2 uses Antibody Designs).\n    for ws in wb:\n        if norm(ws.title)==\'antibodydesigns\' and norm(ws.cell(3,15).value).startswith(\'construct1sequence\'):\n            return ws,\'v42\'\n    for ws in wb:\n        if \'sequence\' in ws.title.lower():\n            heads=[norm(ws.cell(1,i).value) for i in range(1,min(ws.max_column,45)+1)]\n            if any(\'vhvdjsequence\' in h for h in heads) or any(\'construct1sequence\' in h for h in heads):return ws,\'legacy\'\n    if \'Data entry\' in wb.sheetnames and norm(wb[\'Data entry\'].cell(3,6).value)==\'vhvdjsequencednaoraa\':\n        return wb[\'Data entry\'],\'ec\'\n    raise ValueError(\'Cannot find supported worksheet (v4.2, EC Data entry or legacy Sequence).\')\n\ndef parse(wb):\n    ws,kind=find_input(wb)\n    rows=[]\n    if kind==\'v42\':\n        for r in range(4,ws.max_row+1):\n            name=ws.cell(r,3).value; first=seq(ws.cell(r,15).value); second=seq(ws.cell(r,22).value)\n            if not populated(name) or not first:continue\n            if seq(ws.cell(r,30).value) or seq(ws.cell(r,37).value):raise ValueError(f\'Row {r}: 3 or 4 constructs unsupported\')\n            rows.append(dict(row=r,name=str(name).strip(),pur=ws.cell(r,4).value,tag=ws.cell(r,6).value,iso=ws.cell(r,7).value,well=ws.cell(r,2).value,plate=ws.cell(r,1).value,chains=[{\'name\':ws.cell(r,9).value,\'sequence\':first,\'vector\':ws.cell(r,10).value,\'kozak\':ws.cell(r,12).value,\'leader\':ws.cell(r,14).value}]+([{\'name\':ws.cell(r,16).value,\'sequence\':second,\'vector\':ws.cell(r,17).value,\'kozak\':ws.cell(r,19).value,\'leader\':ws.cell(r,21).value}] if second else [])))\n    elif kind==\'ec\':\n        h={norm(ws.cell(3,c).value):c for c in range(1,ws.max_column+1)}\n        def get(r,key):return ws.cell(r,h[key]).value if key in h else None\n        for r in range(4,ws.max_row+1):\n            name=get(r,\'iggname\'); first=seq(get(r,\'vhvdjsequencednaoraa\'));second=seq(get(r,\'vlsequencednaoraa\'))\n            if not populated(name) or not first:continue\n            fmt=get(r,\'format\')\n            rows.append(dict(row=r,name=str(name).strip(),pur=None,tag=None,iso=fmt,well=get(r,\'well\'),plate=get(r,\'plate\'),chains=[{\'name\':get(r,\'vhname\'),\'sequence\':first,\'vector\':get(r,\'vhvectornameinsertionsitename\'),\'kozak\':get(r,\'vhkozaksequencedna\'),\'leader\':get(r,\'vhleadersequencedna\')}]+([{\'name\':get(r,\'vlname\'),\'sequence\':second,\'vector\':get(r,\'vlvectornameinsertionsitename\'),\'kozak\':get(r,\'vlkozaksequencedna\'),\'leader\':get(r,\'vlleadersequencedna\')}] if second else [])))\n    else:\n        h={norm(ws.cell(1,c).value):c for c in range(1,ws.max_column+1)}\n        def get(r,key):return ws.cell(r,h[key]).value if key in h else None\n        for r in range(2,ws.max_row+1):\n            name=get(r,\'iggname\') or get(r,\'antibodyname\'); first=seq(get(r,\'vhvdjsequencednaoraa\') or get(r,\'construct1sequencednaoraa\'));second=seq(get(r,\'vlsequencednaoraa\') or get(r,\'construct2sequencednaoraa\'))\n            if not populated(name) or not first:continue\n            rows.append(dict(row=r,name=str(name).strip(),pur=get(r,\'purification\'),tag=get(r,\'histagsequence\'),iso=get(r,\'iggisotype\'),well=get(r,\'well\'),plate=get(r,\'plate\'),chains=[{\'name\':get(r,\'vhname\'),\'sequence\':first,\'vector\':get(r,\'vhvectornameinsertionsitename\'),\'kozak\':get(r,\'vhkozaksequencedna\'),\'leader\':get(r,\'vhleadersequencedna\')}]+([{\'name\':get(r,\'vlname\'),\'sequence\':second,\'vector\':get(r,\'vlvectornameinsertionsitename\'),\'kozak\':get(r,\'vlkozaksequencedna\'),\'leader\':get(r,\'vlleadersequencedna\')} ] if second else [])))\n    if not rows:raise ValueError(\'No antibody rows with a sequence found.\')\n    return rows,ws.title\n\n\ndef parse_csv_rows(source_bytes):\n    """CSV export: each construct is a separate row; pair by IgG Name and validate roles."""\n    sample=source_bytes.decode(\'utf-8-sig\')\n    reader=csv.DictReader(io.StringIO(sample))\n    required={\'IgG Name\',\'Construct Name (DNA)\',\'Insert Sequence (DNA)\'}\n    if not reader.fieldnames or not required.issubset(set(reader.fieldnames)):\n        raise ValueError(\'CSV not recognized: missing \'+\', \'.join(sorted(required-set(reader.fieldnames or []))))\n    groups=collections.OrderedDict()\n    for line,rec in enumerate(reader,start=2):\n        name=(rec.get(\'IgG Name\') or \'\').strip(); sequence=seq(rec.get(\'Insert Sequence (DNA)\'))\n        if not name or not sequence: continue\n        cname=(rec.get(\'Construct Name (DNA)\') or \'\').strip()\n        lower=cname.lower()\n        vh=bool(re.search(r\'(?<![a-z])(?:vh|heavy)(?![a-z])\',lower))\n        vl=bool(re.search(r\'(?<![a-z])(?:vl|light)(?![a-z])\',lower))\n        if vh==vl:raise ValueError(f\'CSV line {line}: cannot identify uniquely heavy versus light chain from {cname!r}\')\n        chain={\'name\':cname,\'sequence\':sequence,\'vector\':rec.get(\'Vector Name\'),\'kozak\':rec.get(\'Kozak Sequence (DNA)\'),\'leader\':rec.get(\'Leader Sequence (DNA)\')}\n        key=(rec.get(\'Plate (fixed plate maps)\') or \'\').strip(),(rec.get(\'Well (fixed plate maps)\') or \'\').strip(),name\n        entry=groups.setdefault(name, {\'row\':line,\'name\':name,\'pur\':None,\'tag\':None,\'iso\':\'hIgG1\' if \'IgG\' in (rec.get(\'Product Type\') or \'\') else None,\'well\':key[1], \'plate\':key[0], \'_chain_map\':{}})\n        if entry[\'well\']!=key[1] or entry[\'plate\']!=key[0]:raise ValueError(f\'{name}: inconsistent source plate/well coordinates\')\n        role=\'VH\' if vh else \'VL\'\n        if role in entry[\'_chain_map\']:raise ValueError(f\'{name}: multiple {role} rows\')\n        entry[\'_chain_map\'][role]=chain\n    rows=[]\n    for name,entry in groups.items():\n        if set(entry[\'_chain_map\'])!={\'VH\',\'VL\'}:raise ValueError(f\'{name}: require exactly one VH and one VL row\')\n        entry[\'chains\']=[entry[\'_chain_map\'][\'VH\'],entry[\'_chain_map\'][\'VL\']]\n        del entry[\'_chain_map\'];rows.append(entry)\n    if not rows:raise ValueError(\'No paired IgG rows in CSV\')\n    return rows,\'CSV / paired VH and VL construct rows\'\n\ndef mode(rows,filename,wb):\n    lengths=set(map(lambda r:len(r[\'chains\']),rows))\n    if len(lengths)>1:raise ValueError(\'Mixed one- and two-chain rows: split inputs before conversion.\')\n    chains=lengths.pop(); pur=[str(r[\'pur\'] or \'\').lower() for r in rows if populated(r[\'pur\'])]\n    found=set(\'nickel\' if (\'his\' in p or \'nickel\' in p or \'ni-\' in p) else \'none\' if (\'supe\' in p or \'supernatant\' in p or \'none\' in p or \'no purification\' in p) else \'protein-a\' if (\'protein a\' in p or \'proa\' in p) else \'unknown\' for p in pur)-{\'unknown\'}\n    if len(found)>1:raise ValueError(\'Mixed purification methods; split before conversion.\')\n    hints=filename.lower()\n    if not found:\n        if re.search(r\'(^|\\W)(his|nickel|ni)(\\W|$)\',hints):found={\'nickel\'}\n        elif \'supe\' in hints or \'none\' in hints:found={\'none\'}\n        elif \'proa\' in hints or \'protein\' in hints:found={\'protein-a\'}\n    # Never infer a purification from a generic sample template. Explicit fallback is reported.\n    purification=next(iter(found)) if found else (\'protein-a\' if chains==2 else \'none\')\n    settings=\' \'.join(str(c.value or \'\') for s in wb if \'settings\' in s.title.lower() for row in s.iter_rows(min_row=1,max_row=min(s.max_row,18),max_col=min(s.max_column,3)) for c in row).lower() if wb is not None else \'\'\n    container=\'tube\' if (re.search(r\'\\btubes?\\b\',hints) or (\'shipping container\' in settings and re.search(r\'\\btubes?\\b\',settings) and \'96-well\' not in settings)) else \'plate\'\n    return chains,purification,container,not bool(found)\n\ndef dropdown_choices(workbook, worksheet, coordinate):\n    """Read exact strings from the XLSX\'s own list validation (named ranges or literals)."""\n    from openpyxl.utils.cell import range_boundaries\n    cell=worksheet[coordinate]\n    for dv in worksheet.data_validations.dataValidation:\n        if dv.type!=\'list\' or not any(cell.coordinate in worksheet[rng] if False else _covers(rng,cell.row,cell.column) for rng in dv.sqref.ranges):\n            continue\n        formula=(dv.formula1 or \'\').strip()\n        if formula.startswith(\'=\'):formula=formula[1:]\n        if formula.startswith(\'"\') and formula.endswith(\'"\'):\n            return formula[1:-1].split(\',\')\n        defined=workbook.defined_names.get(formula)\n        if defined is not None:\n            vals=[]\n            for name,rect in defined.destinations:\n                target=workbook[name.strip("\'")]\n                area=target[rect]\n                if not isinstance(area,tuple):area=((area,),)\n                elif area and not isinstance(area[0],tuple):area=(area,)\n                vals.extend(x.value for row in area for x in row if x.value is not None)\n            return vals\n        if \'!\' in formula:\n            name,rect=formula.rsplit(\'!\',1)\n            area=workbook[name.strip("\'")][rect]\n            if not isinstance(area,tuple):area=((area,),)\n            elif area and not isinstance(area[0],tuple):area=(area,)\n            return [x.value for row in area for x in row if x.value is not None]\n    return None\n\ndef _covers(rng,row,col):\n    return rng.min_row<=row<=rng.max_row and rng.min_col<=col<=rng.max_col\n\ndef match_dropdown(value,choices):\n    """Return an exact dropdown choice if confident; otherwise return the input verbatim."""\n    import difflib\n    if not choices:return value,\'no validation\'\n    if value in choices:return value,\'exact\'\n    if value is None or not str(value).strip():return value,\'empty\'\n    actual=[c for c in choices if norm(c)==norm(value)]\n    if len(actual)==1:return actual[0],\'normalized\'\n    # Ignore generic placeholders; require a very close unique match.\n    candidates=[(difflib.SequenceMatcher(None,str(value).casefold(),str(c).casefold()).ratio(),c)\n                for c in choices if \'custom - enter vector name\' not in str(c).casefold()]\n    candidates.sort(key=lambda x:x[0],reverse=True)\n    if candidates and candidates[0][0]>=0.92 and (len(candidates)==1 or candidates[0][0]-candidates[1][0]>=0.07):\n        return candidates[0][1],\'fuzzy\'\n    return value,\'verbatim\'\n\ndef extend_validations(worksheet,last_row):\n    """Templates validate rows 4:18; extend the existing rules to cover all populated output rows."""\n    from openpyxl.worksheet.cell_range import CellRange,MultiCellRange\n    for dv in worksheet.data_validations.dataValidation:\n        expanded=[]\n        for region in dv.sqref.ranges:\n            cr=CellRange(str(region))\n            if cr.min_row==4 and cr.max_row<last_row:\n                cr.max_row=last_row\n            expanded.append(str(cr))\n        dv.sqref=MultiCellRange(\' \'.join(expanded))\n\ndef convert_to_v42(rows, source_name, template_bytes):\n    """Map normalized EC or paired-row CSV into original v4.2 workbook structure."""\n    import difflib\n    dest=openpyxl.load_workbook(io.BytesIO(template_bytes))\n    ws=dest[\'Antibody Designs\']\n    # Source v4.2 was a filled-in sample: clear customer-specific contact and assay selections.\n    if \'Contact Information\' in dest:\n        contact=dest[\'Contact Information\']\n        for rr in range(4,contact.max_row+1):\n            field=str(contact.cell(rr,1).value or \'\').lower()\n            if any(k in field for k in (\'name\',\'company\',\'department\',\'address\',\'city\',\'state\',\'zip\',\'country\',\'phone\',\'email\',\'shipping\',\'billing\',\'purchase order\',\'customer\',\'contact\')):\n                contact.cell(rr,2).value=None\n    if \'Antibody Settings\' in dest:\n        settings=dest[\'Antibody Settings\']\n        for rr in range(3,settings.max_row+1):\n            if settings.cell(rr,1).value and settings.cell(rr,2).data_type!=\'f\':settings.cell(rr,2).value=None\n    if \'Characterization Add-ons\' in dest:\n        addon=dest[\'Characterization Add-ons\']\n        for row in addon.iter_rows(min_row=3):\n            for cell in row[2:]:\n                if cell.data_type!=\'f\' and cell.value is not None and (str(cell.value).strip().lower()==\'yes\' or (cell.column==3 and str(cell.value).strip().lower() in (\'no\',\'n/a\'))):cell.value=None\n    warnings=[]\n    def choice(r,col,v):\n        if v is None:return None\n        opts=dropdown_choices(dest,ws,f\'{get_column_letter(col)}4\')\n        result,status=match_dropdown(v,opts)\n        if status==\'verbatim\' and opts:\n            warnings.append(f\'{r}: column {get_column_letter(col)}: original {v!r} preserved (not in dropdown)\')\n        return result\n    # Clear only actual antibody data. Preserve formula scaffolding and workbook dropdowns.\n    for r in range(4,ws.max_row+1):\n        for c in [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37]:\n            cell=ws.cell(r,c)\n            if cell.data_type!=\'f\':cell.value=None\n    for index,record in enumerate(rows):\n        r=index+4\n        if r>ws.max_row:raise ValueError(\'v4.2 maximum number of rows exceeded\')\n        cs=record[\'chains\']\n        pur=record[\'pur\']\n        if not populated(pur):\n            hints=source_name.lower()\n            pur=(\'Nickel\' if \'nickel\' in hints or re.search(r\'\\bhis\\b\',hints) else \'None\' if \'none\' in hints or \'supe\' in hints else \'Protein A\' if (\'proa\' in hints or \'protein-a\' in hints) else None)\n            if not pur:warnings.append(f\'{record["name"]}: purification not represented in EC/sequence input; left blank\')\n        fmt=str(record[\'iso\'] or \'\').strip()\n        if fmt==\'VHH-Fc-IgG1\':fmt=\'hIgG1 VHH Fc Fusion\'\n        elif fmt in (\'hIgG1\',\'hIgG2\',\'hIgG4\'):pass\n        elif fmt==\'Fab\':fmt=\'Fab\'\n        else:\n            if fmt:warnings.append(f\'{record["name"]}: isotype {fmt!r} copied or best matched; review\')\n        fields={1:record[\'plate\'],2:record[\'well\'],3:record[\'name\'],4:pur,7:fmt,\n                9:cs[0][\'name\'],10:cs[0][\'vector\'],12:cs[0][\'kozak\'],14:cs[0][\'leader\'],15:cs[0][\'sequence\']}\n        if len(cs)==2:fields.update({16:cs[1][\'name\'],17:cs[1][\'vector\'],19:cs[1][\'kozak\'],21:cs[1][\'leader\'],22:cs[1][\'sequence\']})\n        for col,value in fields.items():\n            if value is not None:ws.cell(r,col).value=choice(record[\'name\'],col,value)\n        # Set Kozak / leader manual control in v4.2 to avoid conflicting implied built-ins.\n        for col,sequence_col,setting in ((11,12,\'Custom\'),(13,14,\'Custom\'),(18,19,\'Custom\'),(20,21,\'Custom\')):\n            if len(cs)==1 and col in (18,20):continue\n            if ws.cell(r,sequence_col).value:\n                opts=dropdown_choices(dest,ws,f\'{get_column_letter(col)}4\') or []\n                match=next((v for v in opts if \'custom\' in str(v).lower()),None)\n                if match:ws.cell(r,col).value=match\n                else:warnings.append(f\'{record["name"]}: no verified custom selection for {get_column_letter(col)}; setting left blank\')\n        if r!=4:\n            for c in range(1,38):\n                if not ws.cell(r,c).has_style:ws.cell(r,c)._style=copy.copy(ws.cell(4,c)._style)\n    notes=f\'Input: {source_name}\\nOutput: v4.2\\nAntibodies: {len(rows)}\\nWarnings: {len(warnings)}\\n\'+\'\\n\'.join(warnings)\n    out=io.BytesIO();dest.save(out)\n    check=openpyxl.load_workbook(io.BytesIO(out.getvalue()),read_only=True)[\'Antibody Designs\']\n    assert all(check.cell(i+4,15).value==v[\'chains\'][0][\'sequence\'] for i,v in enumerate(rows))\n    assert all(check.cell(i+4,22).value==v[\'chains\'][1][\'sequence\'] for i,v in enumerate(rows) if len(v[\'chains\'])==2)\n    return out.getvalue(),notes,dict(rows=len(rows),warnings=warnings,direction=\'to_v42\')\n\ndef convert_routed(source_bytes,source_name,zip_bytes,v42_template_bytes):\n    """Returns list of (output suffix, XLSX bytes, report text, metadata)."""\n    if source_name.lower().endswith(\'.csv\'):\n        rows,_=parse_csv_rows(source_bytes)\n        outputs=[]\n        ec_data,ec_report,ec_meta=convert_bytes(source_bytes,source_name,zip_bytes)\n        outputs.append((\'_EC\',ec_data,ec_report,ec_meta))\n        v42_data,v42_report,v42_meta=convert_to_v42(rows,source_name,v42_template_bytes)\n        outputs.append((\'_v4.2\',v42_data,v42_report,v42_meta))\n        return outputs\n    wb=openpyxl.load_workbook(io.BytesIO(source_bytes))\n    ws,kind=find_input(wb)\n    if kind==\'ec\':\n        rows,_=parse(wb)\n        data,report,meta=convert_to_v42(rows,source_name,v42_template_bytes)\n        return [(\'_v4.2\',data,report,meta)]\n    if kind==\'v42\':\n        data,report,meta=convert_bytes(source_bytes,source_name,zip_bytes)\n        return [(\'_EC\',data,report,meta)]\n    # Older legacy Sequence forms are routed to EC for backward compatibility.\n    data,report,meta=convert_bytes(source_bytes,source_name,zip_bytes)\n    return [(\'_EC\',data,report,meta)]\n\ndef convert_bytes(source_bytes, source_name, zip_bytes):\n    if source_name.lower().endswith(\'.csv\'):\n        source=None; rows,sheet=parse_csv_rows(source_bytes)\n    else:\n        source=openpyxl.load_workbook(io.BytesIO(source_bytes));rows,sheet=parse(source)\n    chains,pur,container,defaulted=mode(rows,source_name,source)\n    archive=zipfile.ZipFile(io.BytesIO(zip_bytes)); group=\'igg\' if chains==2 else \'vhh\'\n    matches=[n for n in archive.namelist() if f\'-{group}-{pur}-\' in n.lower() and (\'_plate\' in n.lower() if container==\'plate\' else (\'_tube\' in n.lower()))]\n    if len(matches)!=1:raise ValueError(f\'Expected 1 template for {group}/{pur}/{container}, found {matches}\')\n    dest=openpyxl.load_workbook(io.BytesIO(archive.read(matches[0])));ws=dest[\'Data entry\'];hdr={norm(ws.cell(3,c).value):c for c in range(1,ws.max_column+1)}\n    warnings=[];counts=collections.Counter()\n    # Inspect original validation definitions before extending; names resolve to exact hidden list values.\n    validation_columns={}\n    for col in range(1,ws.max_column+1):\n        opts=dropdown_choices(dest,ws,f\'{get_column_letter(col)}4\')\n        if opts is not None:validation_columns[col]=opts\n    def put(r,alias,value,record_name=None):\n        if value is None:return\n        for a in alias:\n            if norm(a) in hdr:\n                col=hdr[norm(a)]\n                if col in validation_columns:\n                    exact,status=match_dropdown(value,validation_columns[col])\n                    if status==\'verbatim\':\n                        warnings.append(f\'{record_name or "Row "+str(r)}: {ws.cell(3,col).value}: {value!r} not confidently matched to dropdown; retained original input verbatim (outside validation list)\')\n                        if \'vector\' in str(ws.cell(3,col).value).lower():counts[\'verbatim_vectors\']+=1\n                    if status in (\'normalized\',\'fuzzy\'):counts[\'dropdown_corrected\']+=1\n                    value=exact\n                ws.cell(r,col).value=value\n                return\n    # Clear all bundled example rows and extend them, retaining template structure.\n    example=4\n    for r in range(4,max(ws.max_row+1,len(rows)+4)):\n        for c in range(1,ws.max_column+1):\n            cell=ws.cell(r,c)\n            if r>example and r<len(rows)+4:\n                model=ws.cell(example,c)\n                if not cell.has_style:cell._style=copy.copy(model._style)\n            cell.value=None\n    for idx,record in enumerate(rows):\n        r=idx+4;cs=record[\'chains\'];ch=cs[0]\n        well=record[\'well\'] if populated(record[\'well\']) else f\'{chr(65+(idx%96)%8)}{1+(idx%96)//8}\'\n        plate=record[\'plate\'] if populated(record[\'plate\']) else 1+idx//96\n        if isinstance(well,str) and not re.match(r\'^[A-H](?:[1-9]|1[0-2])$\',well.strip(),re.I):warnings.append(f\'{record["name"]}: unrecognized well {well!r}\')\n        put(r,[\'Plate #\'],plate);put(r,[\'Well\'],well);put(r,[\'Plate label\'],f\'Plate {plate}\')\n        put(r,[\'IGG Name\'],record[\'name\']);put(r,[\'VH Name\'],ch[\'name\'] or record[\'name\']+\'_VH\');put(r,[\'VH (VDJ) Sequence (DNA or AA)\'],ch[\'sequence\'])\n        fmt=str(record[\'iso\'] or \'\').strip()\n        if chains==1:\n            if \'vhh\' in fmt.lower() and \'fc\' in fmt.lower():fmt=\'VHH-Fc-IgG1\'\n            elif \'scfv\' in fmt.lower():fmt=\'scFv-His\' if pur==\'nickel\' else \'scFv\'\n            elif \'vhh\' in fmt.lower():fmt=\'VHH-His\' if pur==\'nickel\' else \'VHH-Fc-IgG1\' if pur==\'protein-a\' else \'VHH\'\n            elif not fmt:fmt=\'VHH-His\' if pur==\'nickel\' else \'VHH-Fc-IgG1\' if pur==\'protein-a\' else \'Other\'\n        else:\n            if \'fab\' in source_name.lower():fmt=\'Fab\'\n            elif \'fab\' in fmt.lower():fmt=\'Fab\'\n            elif norm(fmt) in (\'higg1\',\'higg2\',\'higg4\'):pass\n            elif not fmt:fmt=\'Other\';warnings.append(f\'{record["name"]}: no exact IgG format; set Other\')\n            elif fmt not in (\'hIgG1\',\'hIgG2\',\'hIgG4\',\'Fab\',\'Other\'):fmt=\'Other\';warnings.append(f\'{record["name"]}: unrecognized isotype; set Other\')\n        put(r,[\'Format\'],fmt,record[\'name\'])\n        put(r,[\'His-tag length\'],len(re.findall(\'CAC\',str(record[\'tag\'] or \'\').upper())) if record[\'tag\'] and re.fullmatch(r\'(CAC|CAT)+\',str(record[\'tag\']).upper()) else 6 if pur==\'nickel\' else None)\n        for aliases,val in [([\'VH Vector Name - Insertion Site Name\'],ch[\'vector\']),([\'VH Kozak Sequence (DNA)\'],ch[\'kozak\']),([\'VH Leader Sequence (DNA)\'],ch[\'leader\'])]:put(r,aliases,val,record[\'name\'])\n        if chains==2:\n            vl=cs[1];put(r,[\'VL Name\'],vl[\'name\'] or record[\'name\']+\'_VL\');put(r,[\'VL Sequence (DNA or AA)\'],vl[\'sequence\'])\n            for a,v in [([\'VL Vector Name - Insertion Site Name\'],vl[\'vector\']),([\'VL Kozak Sequence (DNA)\'],vl[\'kozak\']),([\'VL Leader Sequence (DNA)\'],vl[\'leader\'])]:put(r,a,v,record[\'name\'])\n        counts[\'rows\']+=1;counts[\'chains\']+=chains\n    if source_name.lower().endswith(\'.csv\'):\n        audit=dest.create_sheet(\'Original construct vectors\')\n        audit.append([\'IgG Name\',\'Chain\',\'Construct Name\',\'Original Vector Name\',\'Destination dropdown selection\',\'Input CSV line\',\'Notes\'])\n        for record in rows:\n            for i,ch in enumerate(record[\'chains\']):\n                a=\'VH Vector Name - Insertion Site Name\' if i==0 else \'VL Vector Name - Insertion Site Name\'\n                final=ws.cell(rows.index(record)+4,hdr[norm(a)]).value\n                audit.append([record[\'name\'],\'VH\' if i==0 else \'VL\',ch[\'name\'],ch[\'vector\'],final,record[\'row\'],\'Original preserved verbatim; not a valid dropdown choice\' if final not in (dropdown_choices(dest,ws,f\'{get_column_letter(hdr[norm(a)])}4\') or []) else \'\'])\n        audit.freeze_panes=\'A2\';audit.auto_filter.ref=f\'A1:G{audit.max_row}\'\n        for col,width in {\'A\':16,\'B\':10,\'C\':39,\'D\':46,\'E\':50,\'F\':15,\'G\':70}.items(): audit.column_dimensions[col].width=width\n        warnings.insert(0,\'CSV has no purification field: selected Protein A by 2-chain fallback; confirm before ordering\' if defaulted else \'Confirm purification selected from input\')\n    extend_validations(ws,len(rows)+3)\n    notes=f\'Input: {source_name}\\nWorksheet: {sheet}\\nTemplate: {matches[0]}\\nRows: {len(rows)}; constructs per row: {chains}\\nPurification: {pur}\'+(\' (FALLBACK; review)\' if defaulted else \'\')+f\'\\nContainer: {container}\\nDropdown values auto-corrected: {counts["dropdown_corrected"]}; verbatim vectors: {counts["verbatim_vectors"]}\\nWarnings: {len(warnings)}\\n\'+\'\\n\'.join(warnings)\n    out=io.BytesIO();dest.save(out)\n    check=openpyxl.load_workbook(io.BytesIO(out.getvalue()),read_only=True)[\'Data entry\']; assert sum(bool(check.cell(r,hdr[\'vhvdjsequencednaoraa\']).value) for r in range(4,len(rows)+4))==len(rows)\n    return out.getvalue(),notes,dict(rows=len(rows),constructs=chains,purification=pur,container=container,template=matches[0],warnings=warnings,dropdown_corrected=counts[\'dropdown_corrected\'],verbatim_vectors=counts[\'verbatim_vectors\'])\n\n', 'v3_engine': '"""v4.2 to v3.1 converter: retain template XML rather than rebuilding Excel styling."""\nfrom pathlib import Path\nfrom copy import deepcopy\nimport argparse, csv, difflib, io, json, re, zipfile\nfrom lxml import etree as ET\nfrom openpyxl import load_workbook\nfrom openpyxl.utils import get_column_letter\n\nVERSION = \'2.01\'\nTEMPLATE = Path(__file__).parent / \'templates/v3.1.xlsx\'\nNS = \'http://schemas.openxmlformats.org/spreadsheetml/2006/main\'\nQ = lambda name: f\'{{{NS}}}{name}\'\nMAP = {\'A\':\'A\', \'B\':\'B\', \'C\':\'C\', **{get_column_letter(i):get_column_letter(i-3) for i in range(7,23)}}\nGROUPS = {\'D\':\'Isotype\',\'E\':\'Light Chain (λ or κ)\',\'G\':\'VH Vector Name - Insertion Site Name\',\'H\':\'Kozak Setting\',\'I\':\'Kozak Sequence\',\'J\':\'Leader Setting\',\'K\':\'Leader Sequence\',\'N\':\'VL Vector Name - Insertion Site Name\',\'O\':\'Kozak Setting\',\'P\':\'Kozak Sequence\',\'Q\':\'Leader Setting\',\'R\':\'Leader Sequence\'}\nORDER_CATEGORIES = {3:\'Scale*\',4:\'Pooling\',5:\'Cell Line\',6:\'Purification\',7:\'Elution Buffer*\',8:\'Buffer Exchange\',9:\'Normatlization\',10:\'Post Purification Sterile Filtration\',11:\'Aliquots\',12:\'Shipping Container*\',13:\'Custom Plate Map\',14:\'Restriction Enzyme Sites to Avoid for DNA Optimization\',15:\'Please indicate if you prefer not to receive physical samples and only data\',16:\'Ship with testing pending\'}\nALIASES = {3:[\'scale\',\'production scale\',\'expression scale\'],4:[\'pooling\'],5:[\'cell line\',\'expression host\'],6:[\'purification\',\'purification method\'],7:[\'elution buffer\'],8:[\'buffer exchange\'],9:[\'normalization\',\'normatlization\'],10:[\'post purification sterile filtration\',\'sterile filtration\'],11:[\'aliquoting\',\'aliquots\'],12:[\'shipping container\',\'shipping format\'],13:[\'custom plate map\'],14:[\'restriction enzyme sites to avoid for dna optimization\',\'restriction enzyme sites to avoid for codon optimization\'],15:[\'please indicate if you prefer not to receive physical samples and only data\',\'please indicate if you prefer not to receive any physical samples and only data\'],16:[\'ship with testing pending\',\'ship physical samples with additional service requests pending\'],17:[\'additional requests\']}\n\nALIASES[9].append(\'Normalization (select Scale for options) Please note: we do not normalize or invoice for samples below target concentration\')\nALIASES[11].append(\'Aliquoting (Not applicable when characterization services and testing are added, select Scale for options)\')\nALIASES[12].append(\'Shipping Container* (select Scale for options)\')\n\ndef occupied(v):\n    return v is not None and str(v).strip() != \'\' and \'PLEASE SELECT\' not in str(v).upper()\n\ndef norm(v):\n    return re.sub(r\'[^a-z0-9]+\', \'\', str(v).lower().replace(\'λ\',\'lambda\').replace(\'κ\',\'kappa\'))\n\ndef options():\n    w = load_workbook(TEMPLATE, read_only=True, data_only=True)\n    out = {}; category = None\n    for a,b in w[\'Sheet3\'].iter_rows(max_col=2, values_only=True):\n        if a: category = a\n        if category and b: out.setdefault(category, []).append(b)\n    w.close(); out[\'Cell Line\']=[\'HEK293\',\'CHO\']\n    return out\n\ndef closest(value, choices):\n    """Return actual template spelling; expose nonexact choices for review."""\n    clean = re.sub(r\'^Empty Vector\\s*-\\s*\',\'\',str(value),flags=re.I)\n    choices = [c for c in choices if not c.startswith(\'PLEASE SELECT\')]\n    for c in choices:\n        if norm(clean) == norm(c): return c, \'matched\'\n    ranked = sorted(((difflib.SequenceMatcher(None,norm(clean),norm(c)).ratio(),c) for c in choices),reverse=True)\n    if ranked and ranked[0][0] >= .55:\n        return ranked[0][1], \'closest option; review\'\n    return None, \'no reliable option; original preserved in report\'\n\ndef set_cell(root, address, value):\n    data=root.find(Q(\'sheetData\')); rn=re.search(r\'\\d+\',address).group()\n    row=data.find(f"{Q(\'row\')}[@r=\'{rn}\']")\n    if row is None:\n        row=ET.Element(Q(\'row\'),r=rn)\n        following=next((r for r in data if int(r.get(\'r\'))>int(rn)),None)\n        if following is None: data.append(row)\n        else: data.insert(data.index(following),row)\n    cell=row.find(f"{Q(\'c\')}[@r=\'{address}\']")\n    if cell is None:\n        cell=ET.SubElement(row,Q(\'c\'),r=address)\n        # Copy template input styling from row 4 for added sequence rows.\n        model=data.find(f"{Q(\'row\')}[@r=\'4\']/{Q(\'c\')}[@r=\'{re.sub(r\'[0-9]\', \'\',address)}4\']")\n        if model is not None and model.get(\'s\'): cell.set(\'s\',model.get(\'s\'))\n    for child in list(cell): cell.remove(child)\n    cell.attrib.pop(\'t\',None)\n    if occupied(value):\n        cell.set(\'t\',\'inlineStr\'); text=ET.SubElement(ET.SubElement(cell,Q(\'is\')),Q(\'t\'))\n        text.set(\'{http://www.w3.org/XML/1998/namespace}space\',\'preserve\'); text.text=str(value)\n    # Excel requires cells in column order.\n    def col(c):\n        n=0\n        for ch in re.sub(r\'\\d\',\'\',c.get(\'r\')): n=n*26+ord(ch)-64\n        return n\n    row[:]=sorted(row,key=col)\n\ndef convert(blob):\n    opts=options(); issues=[]; count=0\n    w=load_workbook(io.BytesIO(blob),read_only=True,data_only=True)\n    if \'Antibody Designs\' not in w.sheetnames:\n        w.close(); raise ValueError(\'Missing Antibody Designs tab; expected a v4.2 workbook\')\n    z=zipfile.ZipFile(TEMPLATE); roots={n:ET.fromstring(z.read(n)) for n in z.namelist() if n.startswith(\'xl/worksheets/\') and n.endswith(\'.xml\')}\n    seq=roots[\'xl/worksheets/sheet3.xml\']; details=roots[\'xl/worksheets/sheet4.xml\']\n    def report(location,original,result,status):\n        issues.append({\'location\':location,\'original\':original,\'selected\':result,\'status\':status,\'version\':VERSION})\n    # Clear template placeholders, preserving every style and instructional row.\n    for c in list(seq.find(Q(\'sheetData\')).iter(Q(\'c\'))):\n        if int(re.search(r\'\\d+\',c.get(\'r\')).group())>=4: set_cell(seq,c.get(\'r\'),None)\n    try:\n        for source_row,row in enumerate(w[\'Antibody Designs\'].iter_rows(min_row=4,max_col=37,values_only=True),4):\n            if not occupied(row[2]): continue\n            if not occupied(row[14]):\n                raise ValueError(f\'Antibody Designs row {source_row}: named design has no construct 1 sequence\')\n            if any(occupied(v) for v in row[22:]):\n                raise ValueError(f\'Antibody Designs row {source_row}: bispecific/extra-construct fields cannot be represented in v3.1\')\n            count+=1; target=count+3\n            if target>1048576: raise ValueError(\'Too many designs for Excel\')\n            vals={dest:row[ord(src)-65] for src,dest in MAP.items()}\n            # Use source isotype when representable; retain sample-confirmed mini-binder mapping.\n            two=occupied(vals.get(\'S\')) or occupied(vals.get(\'M\'))\n            original_isotype=vals.get(\'D\')\n            if norm(original_isotype) == \'minibinder\':\n                vals[\'D\']=\'hIgG1\' if two else \'hIgG1 VHH Fc Fusion\'\n                report(f\'Antibody Designs!G{source_row}\',original_isotype,vals[\'D\'],\'mini-binder mapping matching supplied example; review\')\n            elif not occupied(original_isotype):\n                vals[\'D\']=\'hIgG1\' if two else \'hIgG1 VHH Fc Fusion\'\n                report(f\'Antibody Designs!G{source_row}\',None,vals[\'D\'],\'chain-based inference; review\')\n            if not two: vals[\'E\']=\'N/A\'\n            for col,cat in GROUPS.items():\n                v=vals.get(col)\n                if not occupied(v): continue\n                if col==\'D\':\n                    exact=next((choice for choice in opts[cat] if norm(choice)==norm(v)),None)\n                    if exact is not None: vals[col]=exact\n                    else:\n                        vals[col]=\'Other, please specify below\'\n                        report(f\'Antibody Designs!G{source_row}\',v,vals[col],\'isotype not represented in v3.1; original retained in report; review\')\n                    continue\n                # Sequence data is never selected by fuzzy text similarity.\n                if col in (\'I\',\'K\',\'P\',\'R\'):\n                    exact=next((choice for choice in opts[cat] if norm(choice)==norm(v)),None)\n                    if exact is not None: vals[col]=exact\n                    else: report(f\'Antibody Designs!{get_column_letter(ord(col)-64+3)}{source_row}\',v,v,\'custom sequence retained verbatim; review\')\n                    continue\n                selected,status=closest(v,opts[cat])\n                if col in (\'G\',\'N\') and status!=\'matched\':\n                    selected=None\n                if selected is None and col in (\'G\',\'N\'): selected=v; status=\'custom vector retained; review\'\n                vals[col]=selected\n                if status!=\'matched\': report(f\'Antibody Designs!{get_column_letter(ord(col)-64+3) if col!="C" else "C"}{source_row}\',v,selected,status)\n            for col in \'ABCDEFGHIJKLMNOPQRS\': set_cell(seq,f\'{col}{target}\',vals.get(col))\n        # Source field names, never row positions, drive Order Details mapping.\n        for r in range(3,18):\n            set_cell(details,f\'B{r}\',None); set_cell(details,f\'C{r}\',None)\n        labels={}\n        tw=load_workbook(TEMPLATE,read_only=True,data_only=True)\n        for r in range(3,18): labels[r]=[norm(tw[\'Order Details\'].cell(r,1).value)]+[norm(a) for a in ALIASES[r]]\n        tw.close(); seen=set()\n        source=next((w[n] for n in (\'Antibody Settings\',\'Order Details\') if n in w.sheetnames),None)\n        if source is None: report(\'Order Details\',None,None,\'source settings tab absent; all inputs left blank\')\n        else:\n            for cells in source.iter_rows(values_only=True):\n                if not cells or not occupied(cells[0]): continue\n                label=norm(cells[0])\n                if label in (\'antibodycustomizations\',\'orderdetailsrequired\',\'orderdetails\'): continue\n                matches=[r for r,aliases in labels.items() if label in aliases]\n                if not matches:\n                    ranked=sorted((max(difflib.SequenceMatcher(None,label,a).ratio() for a in aa),r) for r,aa in labels.items())\n                    if ranked[-1][0]>=.8 and ranked[-1][0]-ranked[-2][0]>=.08: matches=[ranked[-1][1]]\n                value=cells[1] if len(cells)>1 else None; note=cells[2] if len(cells)>2 else None\n                if not occupied(value) and not occupied(note): continue\n                if not matches:\n                    report(f\'{source.title}:{cells[0]}\',value,None,\'unmapped field; review\'); continue\n                r=matches[0]\n                if r in seen: report(f\'Order Details:{cells[0]}\',value,None,\'duplicate field ignored; review\'); continue\n                seen.add(r)\n                if occupied(value):\n                    if r in ORDER_CATEGORIES:\n                        selected,status=closest(value,opts[ORDER_CATEGORIES[r]])\n                        if status!=\'matched\': report(f\'Order Details!B{r}\',value,selected,status)\n                        set_cell(details,f\'B{r}\',selected)\n                        if selected is None: note=f\'{note or ""} [Original selection: {value}]\'.strip()\n                    else: set_cell(details,f\'B{r}\',value)\n                if occupied(note): set_cell(details,f\'C{r}\',note)\n        if \'Contact Information\' in w.sheetnames:\n            contact=roots[\'xl/worksheets/sheet2.xml\']\n            template_contact=load_workbook(TEMPLATE,read_only=True,data_only=True)\n            positions={}\n            for rr in range(4,30):\n                key=norm(template_contact[\'Contact Information\'].cell(rr,1).value)\n                if occupied(template_contact[\'Contact Information\'].cell(rr,1).value): positions.setdefault(key,[]).append(rr)\n            occurrence={}\n            for row in w[\'Contact Information\'].iter_rows(max_col=2,values_only=True):\n                if not occupied(row[0]): continue\n                label=norm(row[0]); index=occurrence.get(label,0)\n                if label in positions:\n                    occurrence[label]=index+1\n                    if index<len(positions[label]) and occupied(row[1]): set_cell(contact,f\'B{positions[label][index]}\',row[1])\n            template_contact.close()\n        # Refresh cached results for the template\'s SEARCH-based completion formulas.\n        lookup=load_workbook(TEMPLATE,read_only=True,data_only=True)\n        search_values=[str(r[0]).lower() for r in lookup[\'Sheet3\'].iter_rows(min_row=21,max_row=75,min_col=2,max_col=2,values_only=True) if occupied(r[0])]\n        lookup.close()\n        for c in details.iter(Q(\'c\')):\n            f=c.find(Q(\'f\'))\n            if f is not None and f.text and \'SEARCH(Sheet3!$B$21:$B$75,B\' in f.text:\n                address=\'B\'+re.search(r\'\\d+\',c.get(\'r\')).group()\n                target=details.find(f".//{Q(\'c\')}[@r=\'{address}\']")\n                value=\'\'.join(target.itertext()).lower() if target is not None else \'\'\n                cached=c.find(Q(\'v\'))\n                if cached is None: cached=ET.SubElement(c,Q(\'v\'))\n                cached.text=\'1\' if any(v in value for v in search_values) else \'0\'\n        dimension=seq.find(Q(\'dimension\'))\n        dimension.set(\'ref\',f\'A1:S{max(4568,count+3)}\')\n        # Remove standard and extended dropdown controls, retain other validation.\n        for root in roots.values():\n            for el in list(root.iter()):\n                if ET.QName(el).localname==\'dataValidation\' and el.get(\'type\')==\'list\': el.getparent().remove(el)\n            for el in list(root.iter()):\n                if ET.QName(el).localname==\'dataValidations\':\n                    if len(el)==0: el.getparent().remove(el)\n                    else: el.set(\'count\',str(len(el)))\n        out=io.BytesIO()\n        with zipfile.ZipFile(out,\'w\',zipfile.ZIP_DEFLATED) as dest:\n            for entry in z.infolist(): dest.writestr(entry,ET.tostring(roots[entry.filename],xml_declaration=True,encoding=\'UTF-8\',standalone=True) if entry.filename in roots else z.read(entry.filename))\n        return out.getvalue(),count,issues\n    finally: w.close(); z.close()\n', 'routing': '\n# Routing + validation: delegates to the unmodified v1.4 / fast-v3.1 conversion engines.\nimport io,os,csv,json,traceback,zipfile\nfrom pathlib import Path\nVERSION = "2.01"\n\ndef kind_from_source(blob,name):\n    if name.lower().endswith(\'.csv\'):\n        parse_csv_rows(blob)\n        return \'sequence_download\'\n    if not name.lower().endswith(\'.xlsx\'):\n        raise ValueError(\'Only .xlsx and paired-row .csv are supported\')\n    wb=openpyxl.load_workbook(io.BytesIO(blob),read_only=False,data_only=False)\n    try:\n        _,kind=find_input(wb)\n        return {\'v42\':\'v42\',\'ec\':\'ec\',\'legacy\':\'sequence_download\'}[kind]\n    finally: wb.close()\n\ndef make_v3_from_v42(v42_bytes):\n    out,count,issues=convert_v3(v42_bytes)\n    return out, count, [f"v3.1 {issue[\'location\']}: {issue[\'status\']} | original={issue[\'original\']!r}; selected={issue[\'selected\']!r}" for issue in issues]\n\n\ndef validate_workbook(blob, expected_rows, kind):\n    wb=openpyxl.load_workbook(io.BytesIO(blob),read_only=True,data_only=True)\n    try:\n        if kind==\'v3.1\':\n            ws=wb[\'Sequence\']; assert ws.cell(2,3).value==\'IgG Name\'\n            cnt=sum(1 for row in ws.iter_rows(min_row=4,min_col=3,max_col=3,values_only=True) if row[0])\n        elif kind==\'v4.2\':\n            ws=wb[\'Antibody Designs\']\n            cnt=sum(1 for row in ws.iter_rows(min_row=4,min_col=3,max_col=3,values_only=True) if row[0])\n        else:\n            ws=wb[\'Data entry\']\n            hdr={norm(c.value):c.column for c in ws[3] if c.value}\n            assert \'iggname\' in hdr\n            cnt=sum(1 for row in ws.iter_rows(min_row=4,min_col=hdr[\'iggname\'],max_col=hdr[\'iggname\'],values_only=True) if row[0])\n        if cnt!=expected_rows:raise AssertionError(f\'Expected {expected_rows} entries, found {cnt} in {kind}\')\n        return cnt\n    finally:wb.close()\n\ndef sequence_rows(blob,name):\n    if name.lower().endswith(\'.csv\'):\n        return parse_csv_rows(blob)[0]\n    wb=openpyxl.load_workbook(io.BytesIO(blob))\n    try:return parse(wb)[0]\n    finally:wb.close()\n\ndef process_one(blob,name):\n    kind=kind_from_source(blob,name)\n    output=[]\n    if kind==\'v42\':\n        rows=sequence_rows(blob,name)\n        ec,ec_notes,ec_meta=convert_bytes(blob,name,EC_TEMPLATES)\n        output.append((\'EC\',ec,len(rows),ec_meta.get(\'warnings\',[]),ec_notes))\n        v3,count,issues=make_v3_from_v42(blob)\n        output.append((\'v3.1\',v3,count,issues,\'\'))\n    elif kind==\'ec\':\n        rows=sequence_rows(blob,name)\n        v42,notes,meta=convert_to_v42(rows,name,V42_TEMPLATE)\n        output.append((\'v4.2\',v42,len(rows),meta.get(\'warnings\',[]),notes))\n    else:\n        rows=sequence_rows(blob,name)\n        ec,ec_notes,ec_meta=convert_bytes(blob,name,EC_TEMPLATES)\n        output.append((\'EC\',ec,len(rows),ec_meta.get(\'warnings\',[]),ec_notes))\n        v42,v42_notes,v42_meta=convert_to_v42(rows,name,V42_TEMPLATE)\n        output.append((\'v4.2\',v42,len(rows),v42_meta.get(\'warnings\',[]),v42_notes))\n        v3,count,issues=make_v3_from_v42(v42)\n        output.append((\'v3.1\',v3,count,issues,\'Sequence Download via v4.2\'))\n    for label,content,n,warnings,notes in output:\n        validate_workbook(content,n,label)\n    return kind,output\n\ndef run_unified(uploaded, output_dir=\'converted\'):\n    output_dir=Path(output_dir);output_dir.mkdir(parents=True,exist_ok=True)\n    archives=[]\n    for name,blob in uploaded.items():\n        kind,outputs=process_one(blob,name)\n        stem=Path(name.replace(\'\\\\\',\'/\')).stem\n        base=f\'{stem}_converted_v{VERSION}\'\n        report=[];warnings=[];generated=[]\n        for label,content,count,issues,notes in outputs:\n            filename=f\'{base}_{label}.xlsx\'\n            generated.append((filename,content))\n            report.append([name,kind,label,count,len(issues),\'OK\',VERSION])\n            warnings.extend([[name,label,issue] for issue in issues])\n            if notes: generated.append((f\'{base}_{label}_notes.txt\',notes.encode(\'utf-8\')))\n        def csvdata(header,rows):\n            out=io.StringIO();writer=csv.writer(out);writer.writerow(header);writer.writerows(rows)\n            return out.getvalue().encode(\'utf-8-sig\')\n        path=output_dir/f\'{base}.zip\'\n        with zipfile.ZipFile(path,\'w\',zipfile.ZIP_DEFLATED) as archive:\n            for filename,content in generated: archive.writestr(filename,content)\n            archive.writestr(f\'{base}_report.csv\',csvdata([\'Input\',\'Detected format\',\'Output\',\'Antibodies\',\'Warnings\',\'Status\',\'Converter version\'],report))\n            archive.writestr(f\'{base}_warnings.csv\',csvdata([\'Input\',\'Output\',\'Warning\'],warnings))\n        with zipfile.ZipFile(path) as archive:\n            if archive.testzip() is not None: raise ValueError(\'Output ZIP integrity check failed\')\n        archives.append(path)\n    return archives\n', 'v3_template': '...'}
RUNTIME_SOURCE = '"""Initialize isolated unified and v3.1 engines from notebook-embedded source assets."""\nimport base64\nfrom pathlib import Path\n\n\ndef initialize(sources, directory):\n    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)\n    template_dir=directory/\'templates\';template_dir.mkdir(exist_ok=True)\n    (template_dir/\'v3.1.xlsx\').write_bytes(base64.b64decode(sources[\'v3_template\']))\n    namespace={\'__name__\':\'unified_release\',\'__file__\':str(directory/\'unified.py\')}\n    exec(compile(sources[\'assets\'],\'embedded_templates\',\'exec\'),namespace)\n    exec(compile(sources[\'ec_engine\'],\'embedded_ec_engine\',\'exec\'),namespace)\n    # Keep norm/MAP/options and other v3 globals out of the EC engine namespace.\n    v3={\'__name__\':\'unified_v3\',\'__file__\':str(directory/\'v3_engine.py\')}\n    exec(compile(sources[\'v3_engine\'],\'embedded_v3_engine\',\'exec\'),v3)\n    namespace[\'convert_v3\']=v3[\'convert\']\n    exec(compile(sources[\'routing\'],\'embedded_routing\',\'exec\'),namespace)\n    return namespace\n'
_runtime = {'__name__':'unified_runtime'}
exec(RUNTIME_SOURCE, _runtime)
_work_dir = Path(tempfile.mkdtemp(prefix='unified-2-01-'))
_engine = _runtime['initialize'](RELEASE_SOURCES, _work_dir)
(_work_dir / 'colab_ui.py').write_text('"""Colab upload UI with a native file picker and explicit multiple-file drop zone."""\nimport base64\nfrom pathlib import Path\nimport uuid\nfrom IPython.display import HTML, JSON, display\nfrom google.colab import output\n\n\ndef convert_upload(name, encoded, converter, output_dir):\n    """One callback per file keeps a batch from becoming one large request."""\n    try:\n        if not name.lower().endswith((\'.xlsx\',\'.csv\')):\n            raise ValueError(\'Please choose an .xlsx workbook or paired-chain .csv file.\')\n        blob=base64.b64decode(encoded,validate=True)\n        archive=converter({name:blob},output_dir)[0]\n        return {\'name\':archive.name,\'data\':base64.b64encode(archive.read_bytes()).decode(\'ascii\')}\n    except Exception as exc:\n        return {\'error\':str(exc)}\n\n\ndef show_converter(converter, output_dir=\'/content/converted\'):\n    identifier=\'converter_\'+uuid.uuid4().hex\n    callback=identifier+\'.convert\'\n    output.register_callback(callback,lambda name,data:JSON(convert_upload(name,data,converter,output_dir)))\n    markup=Path(__file__).with_name(\'upload.html\').read_text()\n    display(HTML(markup.replace(\'__HOST_ID__\',identifier).replace(\'__CALLBACK__\',callback)))\n    return callback\n')
(_work_dir / 'upload.html').write_text('<div id="__HOST_ID__" style="font-family:Arial,sans-serif;max-width:650px;color:#243342">\n  <h2 style="margin-bottom:6px">Antibody form converter</h2>\n  <p>Unified intake converter · Version 2.01</p>\n  <div class="dropzone" tabindex="0" role="button" aria-label="Choose or drop Excel or CSV files"\n       style="border:2px dashed #7594bc;border-radius:12px;padding:32px;text-align:center;background:#f5f8fc;cursor:pointer">\n    <strong>Drop Excel or CSV files here</strong>\n    <p style="margin:12px">or</p>\n    <button type="button" style="padding:10px 24px;border:0;border-radius:6px;background:#2563eb;color:white;cursor:pointer">Choose files</button>\n    <input type="file" accept=".xlsx,.csv" multiple hidden>\n    <p style="font-size:13px;margin-bottom:0">One or multiple files · Conversion starts automatically</p>\n  </div>\n  <div class="results" aria-live="polite"></div>\n</div>\n<script>\n(() => {\n const host=document.getElementById(\'__HOST_ID__\');\n const zone=host.querySelector(\'.dropzone\'), input=host.querySelector(\'input\');\n const results=host.querySelector(\'.results\');\n let busy=false;\n const read=file=>new Promise((resolve,reject)=>{\n   const reader=new FileReader();\n   reader.onload=()=>resolve(reader.result.split(\',\')[1]);\n   reader.onerror=()=>reject(new Error(\'Could not read this file. Please try again.\'));\n   reader.readAsDataURL(file);\n });\n async function processFiles(selected){\n   if(busy || !selected.length) return;\n   busy=true; zone.setAttribute(\'aria-disabled\',\'true\');\n   zone.querySelector(\'button\').disabled=true; input.disabled=true;\n   results.replaceChildren();\n   try{\n     for(const file of Array.from(selected)){\n       const row=document.createElement(\'div\'); row.style.margin=\'18px 0\';\n       const title=document.createElement(\'strong\'); title.textContent=file.name;\n       const status=document.createElement(\'p\'); status.textContent=\'Converting…\';\n       row.append(title,status); results.append(row);\n       try{\n         if(!/\\.(xlsx|csv)$/i.test(file.name)) throw new Error(\'Please choose an .xlsx workbook or paired-chain .csv file.\');\n         const payload=await read(file);\n         const response=await google.colab.kernel.invokeFunction(\'__CALLBACK__\',[file.name,payload],{});\n         const result=response.data[\'application/json\'];\n         if(result.error) throw new Error(result.error);\n         const bytes=Uint8Array.from(atob(result.data),c=>c.charCodeAt(0));\n         const url=URL.createObjectURL(new Blob([bytes],{type:\'application/zip\'}));\n         const link=document.createElement(\'a\'); link.href=url; link.download=result.name;\n         link.textContent=\'Download ZIP\'; link.style.cssText=\'display:inline-block;padding:10px 20px;background:#18794e;color:white;border-radius:6px;text-decoration:none\';\n         status.textContent=\'Ready. ZIP includes the workbook and review report.\';\n         row.append(link);\n         // Release retained download data when results are replaced by a later batch.\n         row.dataset.downloadUrl=url;\n       }catch(error){status.textContent=\'Could not convert: \'+error.message;}\n     }\n   }finally{\n     busy=false; zone.setAttribute(\'aria-disabled\',\'false\');\n     zone.querySelector(\'button\').disabled=false; input.disabled=false; input.value=\'\';\n   }\n }\n zone.addEventListener(\'click\',event=>{if(!busy && event.target!==input) input.click();});\n zone.addEventListener(\'keydown\',event=>{if(event.target===zone && (event.key===\'Enter\'||event.key===\' \')){event.preventDefault();if(!busy)input.click();}});\n input.addEventListener(\'change\',()=>processFiles(input.files));\n for(const name of [\'dragenter\',\'dragover\',\'dragleave\',\'drop\']){\n   zone.addEventListener(name,event=>{\n     event.preventDefault();event.stopPropagation();\n     zone.style.borderColor=(name===\'dragenter\'||name===\'dragover\')?\'#2563eb\':\'#7594bc\';\n   });\n }\n zone.addEventListener(\'drop\',event=>{if(!busy)processFiles(event.dataTransfer.files);});\n // Keep accidental drops elsewhere in this output from navigating away.\n host.addEventListener(\'dragover\',event=>event.preventDefault());\n host.addEventListener(\'drop\',event=>event.preventDefault());\n const observer=new MutationObserver(records=>{\n   for(const record of records)for(const node of record.removedNodes){\n     if(node.dataset && node.dataset.downloadUrl)URL.revokeObjectURL(node.dataset.downloadUrl);\n   }\n });\n observer.observe(results,{childList:true});\n})();\n</script>\n')
_ui = {'__name__':'unified_ui','__file__':str(_work_dir/'colab_ui.py')}
exec((_work_dir/'colab_ui.py').read_text(), _ui)
clear_output(wait=True)
_controls = _ui['show_converter'](_engine['run_unified'])
